# Hugging Face Models with Gradio
## Beginner-to-Advanced Google Colab Practical

**Project:** Build an interactive conversational AI application using an open-weight Hugging Face model, PyTorch, Transformers, and Gradio.

### Learning outcomes
By completing this practical, students will be able to:
1. Explain the roles of Hugging Face Hub, Transformers, tokenizers, pretrained models, and Gradio.
2. Select a pretrained instruction-tuned language model.
3. Load the tokenizer and model in Google Colab.
4. generate text using `model.generate()`.
5. Build a conversational interface with Gradio.
6. Tune generation parameters such as temperature, top-k, top-p, and maximum output length.
7. Evaluate limitations, resource requirements, and safe-use considerations.

### What you will build
A browser-based chatbot. A user enters a message in Gradio; Python passes the conversation to a Hugging Face tokenizer and model; the model generates a response; Gradio displays it.

**Default model:** `Qwen/Qwen2.5-1.5B-Instruct` (public model; no Hugging Face API token is normally required to download it).

**Hardware:** Select a GPU runtime in Colab. A GPU is strongly recommended. If memory is insufficient, use the smaller `Qwen/Qwen2.5-0.5B-Instruct` model in the configuration cell.

> This is an educational text-generation chatbot, not a verified knowledge system. It can produce incorrect or fabricated information. Do not enter private, confidential, or sensitive data, and do not use it for medical, legal, financial, or other high-stakes decisions.


## 0. Colab setup instructions

1. Open [Google Colab](https://colab.research.google.com/).
2. Select **File → New notebook**.
3. Select **Runtime → Change runtime type → T4 GPU** (or another available GPU).
4. Run the notebook cells from top to bottom.
5. When Gradio starts, open the public `gradio.live` link displayed in the output. The link is temporary and remains active only while the Colab runtime is running.

A free Colab GPU is not guaranteed. If no GPU is available, try again later or use the smaller model/CPU mode for a limited demonstration.


## 1. Check the runtime and install libraries
We use `transformers` to load the model, `torch` for tensor computation, `accelerate` for device placement, and `gradio` for the web interface.

In [ ]:
!nvidia-smi


In [ ]:
!pip -q install -U transformers accelerate gradio sentencepiece


In [ ]:
import torch
import transformers
import gradio as gr

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Gradio:", gr.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The model may load slowly or may not fit in RAM.")


## 2. Choose the model

An instruction-tuned model is trained to respond to user instructions and conversational prompts. Qwen2.5-1.5B-Instruct is a compact model suitable for a teaching demonstration.

- **Recommended:** `Qwen/Qwen2.5-1.5B-Instruct`
- **Lower-memory alternative:** `Qwen/Qwen2.5-0.5B-Instruct`

If you receive a CUDA out-of-memory error, change the model name below to the 0.5B alternative and rerun the model-loading cell.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# If the 1.5B model does not fit, use:
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

SYSTEM_PROMPT = (
    "You are a helpful educational AI assistant. "
    "Explain concepts clearly, use examples when useful, and state when you are unsure."
)


## 3. Load the tokenizer and pretrained model

**Tokenizer:** Converts text into token IDs that the model can process. It also converts generated token IDs back into readable text.

**Model:** A pretrained neural network that predicts the next token repeatedly to produce a response.

`device_map="auto"` lets Accelerate place the model on the available device. On a CUDA GPU, `float16` is commonly used to reduce memory consumption; on CPU, `float32` is selected for compatibility.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if torch.cuda.is_available():
    model_dtype = torch.float16
else:
    model_dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map="auto"
)

model.eval()
print("Model loaded:", MODEL_ID)
print("Input embedding device:", model.get_input_embeddings().weight.device)


## 4. First generation: a simple prompt

This is the simplest end-to-end test before building a chatbot.

1. The tokenizer converts the prompt into tensors.
2. The tensors are passed to `model.generate()`.
3. The model predicts new tokens.
4. We decode only the newly generated tokens so the original prompt is not repeated.


In [ ]:
prompt = "Explain artificial intelligence to a first-year engineering student in three sentences."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(answer)


## 5. Understand the generation parameters

| Parameter | Meaning | Typical effect |
|---|---|---|
| `max_new_tokens` | Maximum number of new tokens | Controls response length |
| `temperature` | Scales next-token probability distribution | Lower values are more focused; higher values are more varied |
| `top_k` | Keeps only the k most likely next tokens | Smaller k restricts choices |
| `top_p` | Keeps the smallest token set whose cumulative probability reaches p | Smaller p makes sampling more focused |
| `do_sample` | Enables sampling | `False` uses greedy/beam-style deterministic decoding depending on other settings |

Temperature, top-k, and top-p affect the distribution from which the next token is selected. They do not add factual knowledge to the model.


In [ ]:
# Compare two temperatures using the same prompt.
comparison_prompt = "Give three creative project ideas using artificial intelligence."

def generate_with_temperature(prompt, temperature):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        ids = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=temperature,
            top_p=0.9
        )
    generated = ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print("Temperature 0.2:\n", generate_with_temperature(comparison_prompt, 0.2))
print("\nTemperature 1.0:\n", generate_with_temperature(comparison_prompt, 1.0))


## 6. Create a reusable chat-generation function

The Qwen chat template formats the system, user, and assistant messages in the structure expected by the model. This is preferable to manually joining conversation strings.

The function below:
- accepts the current user message and prior conversation;
- keeps only the most recent conversation turns to limit context length;
- applies the model's chat template;
- generates a response with adjustable sampling parameters;
- returns readable text.


In [ ]:
def build_messages(message, history, system_prompt):
    messages = [{"role": "system", "content": system_prompt}]

    # Gradio ChatInterface supplies history as message dictionaries
    # in current Gradio versions.
    for item in (history or [])[-8:]:
        if isinstance(item, dict) and item.get("role") in ("user", "assistant"):
            content = item.get("content", "")
            if isinstance(content, str) and content.strip():
                messages.append({"role": item["role"], "content": content})

    messages.append({"role": "user", "content": message})
    return messages


def generate_reply(
    message,
    history,
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    top_k=50
):
    if not message or not message.strip():
        return "Please enter a message."

    messages = build_messages(message, history, system_prompt)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            temperature=max(float(temperature), 0.01),
            top_p=float(top_p),
            top_k=int(top_k),
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    response_ids = generated_ids[0][input_length:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True).strip()

    return response or "The model did not generate a response. Please try again."


## 7. Build the Gradio chatbot interface

Gradio creates a web interface from Python functions. `ChatInterface` handles the chat display and passes the current message and conversation history to `generate_reply`.

The additional controls allow the user to experiment with model behavior without changing Python code.


In [ ]:
with gr.Blocks(title="Hugging Face AI Chatbot") as demo:
    gr.Markdown(
        "# Hugging Face AI Chatbot\n"
        "A conversational demo powered by Qwen2.5 and Transformers. "
        "Responses may be inaccurate; verify important information."
    )

    with gr.Accordion("Generation settings", open=False):
        system_box = gr.Textbox(
            label="System instruction",
            value=SYSTEM_PROMPT,
            lines=2
        )
        max_tokens = gr.Slider(
            minimum=32, maximum=512, value=256, step=32,
            label="Maximum new tokens"
        )
        temperature = gr.Slider(
            minimum=0.1, maximum=1.5, value=0.7, step=0.1,
            label="Temperature"
        )
        top_p = gr.Slider(
            minimum=0.1, maximum=1.0, value=0.9, step=0.05,
            label="Top-p"
        )
        top_k = gr.Slider(
            minimum=1, maximum=100, value=50, step=1,
            label="Top-k"
        )

    chatbot = gr.Chatbot(label="Conversation", height=420, type="messages")
    msg = gr.Textbox(
        label="Your message",
        placeholder="Ask a question and press Enter...",
        lines=2
    )
    clear = gr.Button("Clear conversation")

    def respond(message, history, system, max_new, temp, p, k):
        reply = generate_reply(
            message, history, system, max_new, temp, p, k
        )
        updated = (history or []) + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply}
        ]
        return "", updated

    msg.submit(
        respond,
        inputs=[msg, chatbot, system_box, max_tokens, temperature, top_p, top_k],
        outputs=[msg, chatbot]
    )
    clear.click(lambda: ([], ""), outputs=[chatbot, msg])

demo.launch(share=True, debug=True)


## 8. How to use the application

1. Run the Gradio cell.
2. Wait until the output displays a `https://....gradio.live` link.
3. Open the link in a browser tab.
4. Ask a question, for example:
   - Explain machine learning with a simple example.
   - What is the difference between AI, ML, and deep learning?
   - Write a Python function to check whether a number is prime.
5. Ask a follow-up question to test conversation history.
6. Change temperature, top-p, top-k, and maximum new tokens and compare responses.
7. Use **Clear conversation** to start a new chat.

Keep the Colab tab/runtime active while using the temporary Gradio link. Stop the cell or disconnect the runtime when finished.


## 9. Beginner practical exercises

**Exercise 1 — Model understanding**
1. What is the Hugging Face Hub?
2. What is the difference between a model and a tokenizer?
3. Why do we use an instruction-tuned model?

**Exercise 2 — Prompt experiments**
Run the same prompt with temperature values `0.2`, `0.7`, and `1.2`. Record how the wording and creativity change.

**Exercise 3 — Token limits**
Run the same question with `max_new_tokens` set to `64`, `128`, and `256`. Compare response length and completeness.

**Exercise 4 — Conversation**
Ask the model a question, then ask a follow-up that depends on the earlier answer. Check whether the history is being used.


## 10. Intermediate extension: create a domain-specific tutor

Change the system instruction to define a narrow educational role. For example:

> You are a Data Structures tutor for first-year engineering students. Explain one concept at a time, use a small example, and finish with one practice question.

Then test it with linked-list, stack, and queue questions. This changes the model's behavior through prompting; it does **not** fine-tune the model or guarantee subject-matter accuracy.


In [ ]:
DOMAIN_SYSTEM_PROMPT = (
    "You are a Data Structures tutor for first-year engineering students. "
    "Explain one concept at a time, give a small example, and finish with "
    "one practice question. If uncertain, say so."
)

# Test the domain instruction directly.
test_messages = [
    {"role": "system", "content": DOMAIN_SYSTEM_PROMPT},
    {"role": "user", "content": "Explain a stack with a real-world example."}
]
test_text = tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True
)
test_inputs = tokenizer(test_text, return_tensors="pt").to(model.device)
with torch.inference_mode():
    test_ids = model.generate(
        **test_inputs, max_new_tokens=180,
        do_sample=True, temperature=0.6, top_p=0.9
    )
print(tokenizer.decode(
    test_ids[0][test_inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
))


## 11. Advanced extension: inspect tokenization

This cell shows how a sentence is segmented into tokens and token IDs. A token may be a whole word, part of a word, punctuation, or whitespace-related unit. Tokens are not always equivalent to words.


In [ ]:
sample = "Transformers make pretrained language models easier to use."
encoded = tokenizer(sample)

print("Original text:", sample)
print("Token IDs:", encoded["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("Decoded:", tokenizer.decode(encoded["input_ids"]))
print("Number of tokens:", len(encoded["input_ids"]))


## 12. Advanced extension: compare model sizes

Change `MODEL_ID` to `Qwen/Qwen2.5-0.5B-Instruct`, restart the runtime, and rerun the notebook from the installation/model-loading cells. Compare:
- time taken to load;
- response latency;
- response quality on the same prompts;
- GPU memory usage (when a GPU is available).

Do not load both models simultaneously in a limited-memory Colab runtime.


## 13. Troubleshooting

| Problem | Recommended action |
|---|---|
| CUDA out of memory | Switch to `Qwen/Qwen2.5-0.5B-Instruct`; restart runtime and rerun |
| `torch.cuda.is_available()` is `False` | Select Runtime → Change runtime type → GPU; check availability |
| Model download is slow | Wait for download to finish; avoid restarting the runtime |
| Gradio link does not open | Rerun the Gradio cell and use the newly generated link |
| Package/import error after installation | Runtime → Restart session, then run cells from the top |
| CPU execution is very slow | Use a GPU runtime or the smaller model |
| Model response is inaccurate | Verify independently; improve the prompt, but do not assume prompting guarantees correctness |

If a package upgrade causes an incompatibility, restart the Colab runtime and run the installation cell again. Colab package versions can change over time.


## 14. Student submission requirements

Submit:
1. The completed `.ipynb` notebook.
2. A screenshot of the running Gradio chatbot.
3. A screenshot showing a follow-up question in the same conversation.
4. A short comparison of responses at three temperature settings.
5. Answers to the viva questions below.

### Viva questions
1. What is Hugging Face?
2. What is the Hugging Face Model Hub?
3. What is a pretrained language model?
4. What is tokenization?
5. What is the purpose of `AutoTokenizer`?
6. What does `AutoModelForCausalLM` load?
7. What is the role of PyTorch?
8. What is Gradio?
9. What does `model.generate()` do?
10. What is the difference between temperature and top-p?
11. Why is conversation history included in a chatbot?
12. Does downloading a public model mean that inference is free of compute cost?

### Assessment rubric (suggested: 20 marks)

| Component | Marks |
|---|---:|
| Runtime setup and model loading | 4 |
| Correct text generation | 4 |
| Gradio chatbot implementation | 5 |
| Parameter experiments and observations | 3 |
| Explanation, screenshots, and viva | 4 |
| **Total** | **20** |


## 15. Key concepts recap

- **Hugging Face Hub:** A repository for sharing models, datasets, and related files.
- **Transformers:** A library that provides tools to load and run transformer models.
- **Tokenizer:** Converts text to token IDs and token IDs back to text.
- **Pretrained model:** A model whose weights were learned before the current application is built.
- **Instruction-tuned model:** A model further trained to follow instructions or conversational prompts.
- **Gradio:** A Python library for creating interactive web interfaces for machine-learning applications.
- **Inference:** Using a trained model to generate predictions or responses.

**Project completion:** You have loaded an open-weight model, tested generation, built a Gradio chatbot, exposed generation controls, and completed beginner-to-advanced extension activities.
